# 04b — TCN ECG Forecasting v4 (4.9s → 4.9s)

### Why TCN v3 failed & what's different now

| Problem | TCN v3 | This version |
|---|---|---|
| Loss | HuberLoss(delta=0.5) — clips spike errors | SpikeWeightedMSELoss — 5x spike penalty |
| Decoder | Attention pooling → single vector → MLP | U-Net decoder with skip connections |
| Temporal info | All temporal structure destroyed at bottleneck | Preserved via skip connections |
| Spike tracking | Spike/base ratio ~5-6x (poor) | Target < 3x (excellent) |

The core fix: **Attention pooling collapsed 122 timesteps into a single 128-d vector.
All temporal information was lost.** The MLP decoder had to hallucinate 490×12=5880
values from 128 numbers — impossible for spikes.

This version uses a **U-Net decoder** with skip connections that pass encoder features
directly to the decoder at each resolution level. Spikes in the input are preserved
all the way to the output.

In [1]:
# CELL 1 — IMPORTS
import os, pickle, warnings
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import mean_absolute_error, mean_squared_error

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch : {torch.__version__}")
print(f"Device  : {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
print("✅ Imports ready")

PyTorch : 2.11.0
Device  : cpu
✅ Imports ready


In [2]:
# CELL 2 — LOAD DATA
SAVE_DIR = os.path.join('..', 'data', 'processed')
FIG_DIR  = os.path.join('..', 'reports', 'figures', 'tcn')
CKPT_DIR = os.path.join('..', 'reports', 'checkpoints')
os.makedirs(FIG_DIR,  exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

X_train = np.load(os.path.join(SAVE_DIR, 'X_train.npy'))
y_train = np.load(os.path.join(SAVE_DIR, 'y_train.npy'))
X_val   = np.load(os.path.join(SAVE_DIR, 'X_val.npy'))
y_val   = np.load(os.path.join(SAVE_DIR, 'y_val.npy'))
X_test  = np.load(os.path.join(SAVE_DIR, 'X_test.npy'))
y_test  = np.load(os.path.join(SAVE_DIR, 'y_test.npy'))

with open(os.path.join(SAVE_DIR, 'config.pkl'), 'rb') as f:
    cfg = pickle.load(f)

LEAD_NAMES = cfg['lead_names']
FS         = cfg['sampling_rate']
INPUT_LEN  = cfg['input_len']
HORIZON    = cfg['horizon']
N_LEADS    = cfg['n_leads']

print(f"X_train  : {X_train.shape}   y_train : {y_train.shape}")
print(f"X_val    : {X_val.shape}     y_val   : {y_val.shape}")
print(f"X_test   : {X_test.shape}    y_test  : {y_test.shape}")
print(f"Input    : {INPUT_LEN} samples = {INPUT_LEN/FS:.2f}s")
print(f"Horizon  : {HORIZON}  samples = {HORIZON/FS:.2f}s")
print("✅ Data loaded")

X_train  : (31362, 490, 12)   y_train : (31362, 490, 12)
X_val    : (3920, 490, 12)     y_val   : (3920, 490, 12)
X_test   : (2198, 490, 12)    y_test  : (2198, 490, 12)
Input    : 490 samples = 4.90s
Horizon  : 490  samples = 4.90s
✅ Data loaded


In [3]:
# CELL 3 — DATALOADERS
class ECGForecastDataset(Dataset):
    def __init__(self, X, y, channel_first=False):
        self.X = torch.from_numpy(np.asarray(X, dtype=np.float32))
        self.y = torch.from_numpy(np.asarray(y, dtype=np.float32))
        self.channel_first = channel_first
    def __len__(self): return len(self.X)
    def __getitem__(self, idx):
        x, y = self.X[idx], self.y[idx]
        if self.channel_first: x = x.permute(1, 0)
        return x, y

def make_loaders(X_tr, y_tr, X_v, y_v, X_te, y_te,
                 channel_first=False, batch_train=64, batch_eval=128):
    kw = dict(num_workers=0, pin_memory=(DEVICE.type == 'cuda'))
    tr = DataLoader(ECGForecastDataset(X_tr, y_tr, channel_first),
                    batch_size=batch_train, shuffle=True, drop_last=True, **kw)
    vl = DataLoader(ECGForecastDataset(X_v,  y_v,  channel_first),
                    batch_size=batch_eval,  shuffle=False, **kw)
    te = DataLoader(ECGForecastDataset(X_te, y_te, channel_first),
                    batch_size=batch_eval,  shuffle=False, **kw)
    return tr, vl, te

tcn_tr, tcn_vl, tcn_te = make_loaders(
    X_train, y_train, X_val, y_val, X_test, y_test, channel_first=True)
xb, yb = next(iter(tcn_tr))
print(f"Batch: x={xb.shape}  y={yb.shape}")
print(f"Train batches: {len(tcn_tr)} | Val: {len(tcn_vl)} | Test: {len(tcn_te)}")
print("✅ DataLoaders ready")

Batch: x=torch.Size([64, 12, 490])  y=torch.Size([64, 490, 12])
Train batches: 490 | Val: 31 | Test: 18
✅ DataLoaders ready


In [4]:
# CELL 4 — TCN v4 MODEL: Multi-Scale Encoder + U-Net Decoder
#
# Why TCN v3 failed:
#   Attention pooling collapsed 122 timesteps → 1 vector.
#   ALL temporal information was destroyed.
#   The MLP decoder had to hallucinate 5880 values from 128 numbers.
#
# The fix: U-Net decoder with skip connections
#   Encoder: 490 → 245 → 122 (via MaxPool)
#   Decoder: 122 → 245 → 490 (via ConvTranspose1d)
#   Skip connections pass encoder features to decoder at each resolution.
#   Spikes preserved throughout the network.

class MultiScaleBlock(nn.Module):
    def __init__(self, in_ch, out_ch, dropout=0.2):
        super().__init__()
        mid = out_ch // 3
        self.branch3  = nn.Sequential(
            nn.Conv1d(in_ch, mid, kernel_size=3,  padding=1,  bias=False),
            nn.BatchNorm1d(mid), nn.GELU())
        self.branch7  = nn.Sequential(
            nn.Conv1d(in_ch, mid, kernel_size=7,  padding=3,  bias=False),
            nn.BatchNorm1d(mid), nn.GELU())
        self.branch15 = nn.Sequential(
            nn.Conv1d(in_ch, mid, kernel_size=15, padding=7,  bias=False),
            nn.BatchNorm1d(mid), nn.GELU())
        self.project = nn.Sequential(
            nn.Conv1d(mid*3, out_ch, kernel_size=1, bias=False),
            nn.BatchNorm1d(out_ch), nn.GELU())
        self.residual = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, 1, bias=False),
            nn.BatchNorm1d(out_ch)) if in_ch != out_ch else nn.Identity()
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([self.branch3(x), self.branch7(x), self.branch15(x)], dim=1)
        out = self.drop(self.project(out))
        return F.gelu(out + self.residual(x))


class TCNv4Forecaster(nn.Module):
    """
    Multi-Scale TCN with U-Net decoder for spike-accurate ECG forecasting.

    Encoder:
      in_proj:  12 → 64,  (B, 64, 490)  — saved for skip
      enc1:    64 → 128,  MaxPool(2)    — (B, 128, 245) — saved for skip
      enc2:    128 → 128, MaxPool(2)    — (B, 128, 122)

    Decoder:
      dec1:    128 → 128, ConvTranspose1d(2) + skip from enc1  → (B, 128, 245)
      dec2:    128 → 64,  ConvTranspose1d(2) + skip from in_proj → (B, 64, 490)
      output:  64 → 12 per timestep

    Skip connections preserve spike information at full resolution.
    """
    def __init__(self, n_leads=12, horizon=490, dropout=0.2):
        super().__init__()
        self.horizon = horizon
        self.n_leads = n_leads

        # Encoder
        self.in_proj = nn.Sequential(
            nn.Conv1d(n_leads, 64, kernel_size=1, bias=False),
            nn.BatchNorm1d(64), nn.GELU())  # (B, 64, 490)

        self.enc1 = nn.Sequential(
            MultiScaleBlock(64, 128, dropout=dropout),
            nn.MaxPool1d(2))  # → (B, 128, 245)

        self.enc2 = nn.Sequential(
            MultiScaleBlock(128, 128, dropout=dropout),
            nn.MaxPool1d(2))  # → (B, 128, 122)

        # Bridge: extra conv at lowest resolution
        self.bridge = MultiScaleBlock(128, 128, dropout=dropout)  # (B, 128, 122)

        # Decoder with skip connections
        self.dec1 = nn.Sequential(
            nn.ConvTranspose1d(128, 128, kernel_size=2, stride=2, bias=False),  # → 244
            nn.BatchNorm1d(128), nn.GELU())
        self.dec1_proj = nn.Sequential(
            nn.Conv1d(256, 128, kernel_size=1, bias=False),  # 128+128 from skip
            nn.BatchNorm1d(128), nn.GELU())

        self.dec2 = nn.Sequential(
            nn.ConvTranspose1d(128, 64, kernel_size=2, stride=2, bias=False),  # → 488
            nn.BatchNorm1d(64), nn.GELU())
        # After ConvTranspose1d(2) from 245 → 490 is exact (245*2=490)
        # But from 244 → 490 needs interpolation
        self.dec2_interpolate = True  # flag to handle size mismatch

        self.dec2_proj = nn.Sequential(
            nn.Conv1d(128, 64, kernel_size=1, bias=False),  # 64+64 from skip
            nn.BatchNorm1d(64), nn.GELU())

        # Output: per-timestep projection
        self.output_proj = nn.Sequential(
            nn.Conv1d(64, 64, kernel_size=1, bias=False),
            nn.BatchNorm1d(64), nn.GELU(),
            nn.Conv1d(64, n_leads, kernel_size=1))

    def forward(self, x):
        # Encoder
        skip0 = self.in_proj(x)                    # (B, 64, 490)
        skip1 = self.enc1(skip0)                    # (B, 128, 245)
        enc2  = self.enc2(skip1)                    # (B, 128, 122)
        bridge = self.bridge(enc2)                  # (B, 128, 122)

        # Decoder
        d1 = self.dec1(bridge)                      # (B, 128, 244)
        # Pad to match skip1 (245)
        d1 = F.pad(d1, (0, skip1.shape[2] - d1.shape[2]))  # (B, 128, 245)
        d1 = self.dec1_proj(torch.cat([d1, skip1], dim=1))  # (B, 128, 245)

        d2 = self.dec2(d1)                          # (B, 64, 490)
        # Pad to match skip0 (490) if needed
        if d2.shape[2] != skip0.shape[2]:
            d2 = F.pad(d2, (0, skip0.shape[2] - d2.shape[2]))
        d2 = self.dec2_proj(torch.cat([d2, skip0], dim=1))  # (B, 64, 490)

        out = self.output_proj(d2)                  # (B, 12, 490)
        return out.permute(0, 2, 1)                 # (B, 490, 12)

    def enable_mc_dropout(self):
        self.eval()
        for m in self.modules():
            if isinstance(m, nn.Dropout): m.train()

print("✅ TCNv4Forecaster defined — U-Net decoder with skip connections")

✅ TCNv4Forecaster defined — U-Net decoder with skip connections


In [5]:
# CELL 5 — INSTANTIATE & VERIFY
model    = TCNv4Forecaster(
    n_leads=N_LEADS, horizon=HORIZON, dropout=0.2).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

with torch.no_grad():
    dummy = torch.randn(4, N_LEADS, INPUT_LEN).to(DEVICE)
    out   = model(dummy)
    print(f"✅ Forward pass: input {dummy.shape} → output {out.shape}")
    assert out.shape == (4, HORIZON, N_LEADS), f"Shape mismatch: {out.shape}"

print(f"Total parameters : {n_params:,}")
print(f"Device           : {DEVICE}")
print("✅ Model ready — TCN v4 with U-Net decoder")

✅ Forward pass: input torch.Size([4, 12, 490]) → output torch.Size([4, 490, 12])
Total parameters : 491,136
Device           : cpu
✅ Model ready — TCN v4 with U-Net decoder


In [6]:
# CELL 6 — LOSS + TRAINING ENGINE

# Spike-weighted MSE: spikes get 5x more penalty
class SpikeWeightedMSELoss(nn.Module):
    def __init__(self, spike_weight=5.0, spike_threshold=1.5):
        super().__init__()
        self.spike_weight = spike_weight
        self.spike_threshold = spike_threshold

    def forward(self, pred, target):
        mse = (pred - target) ** 2
        std = target.std(dim=(1, 2), keepdim=True) + 1e-8
        spike_mask = (target.abs() > self.spike_threshold * std).float()
        weight = 1.0 + (self.spike_weight - 1.0) * spike_mask
        return (weight * mse).mean()

criterion = SpikeWeightedMSELoss(spike_weight=5.0, spike_threshold=1.5)
print("✅ SpikeWeightedMSELoss — baseline=1.0x, spike=5.0x penalty")


def train_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0.0
    for xb, yb in tqdm(loader, desc='Train', leave=False, ncols=90):
        xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        pred = model(xb)
        loss = criterion(pred, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * len(xb)
    return total_loss / len(loader.dataset)

@torch.no_grad()
def eval_epoch(model, loader, device):
    model.eval()
    total_loss, preds, targets = 0.0, [], []
    for xb, yb in loader:
        xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
        pred   = model(xb)
        loss   = criterion(pred, yb)
        total_loss += loss.item() * len(xb)
        preds.append(pred.cpu().numpy())
        targets.append(yb.cpu().numpy())
    return total_loss / len(loader.dataset), np.concatenate(preds), np.concatenate(targets)


def train_tcn(model, tr_loader, vl_loader, n_epochs=120, lr=3e-4, patience=20):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=n_epochs, eta_min=1e-6)

    best_val, no_improve = float('inf'), 0
    ckpt    = os.path.join(CKPT_DIR, 'TCN_final.pt')
    history = {'train_loss': [], 'val_loss': [], 'lr': []}

    print(f"\n{'─'*70}")
    print(f"  TCN v4  |  {INPUT_LEN/FS:.1f}s → {HORIZON/FS:.1f}s  |  {n_params:,} params")
    print(f"  MultiScale encoder | U-Net decoder | SpikeWeightedMSE | batch=64")
    print(f"{'─'*70}")

    pbar = tqdm(range(1, n_epochs+1), desc='Epochs', unit='ep', ncols=90)
    for ep in pbar:
        tr_loss       = train_epoch(model, tr_loader, optimizer, DEVICE)
        vl_loss, _, _ = eval_epoch(model, vl_loader, DEVICE)
        current_lr    = optimizer.param_groups[0]['lr']
        scheduler.step()

        history['train_loss'].append(tr_loss)
        history['val_loss'].append(vl_loss)
        history['lr'].append(current_lr)

        is_best = vl_loss < best_val
        if is_best:
            best_val = vl_loss; no_improve = 0
            torch.save(model.state_dict(), ckpt)
        else:
            no_improve += 1

        pbar.set_postfix({'tr': f'{tr_loss:.5f}', 'vl': f'{vl_loss:.5f}',
                          'lr': f'{current_lr:.1e}', 'pat': no_improve})
        if ep % 10 == 0 or is_best:
            tqdm.write(f"  {ep:3d} | train={tr_loss:.5f}  val={vl_loss:.5f}"
                       f"  lr={current_lr:.2e}{'  ★' if is_best else f'  ({no_improve}/{patience})'}")

        if no_improve >= patience:
            tqdm.write(f"  ⏹ Early stopping at epoch {ep}  |  best val={best_val:.6f}")
            break

    model.load_state_dict(torch.load(ckpt, map_location=DEVICE, weights_only=True))
    print(f"\n  Best val loss : {best_val:.6f}")
    print(f"  Checkpoint    : {ckpt}")
    print(f"{'─'*70}\n")
    return history

print("✅ Training engine ready")

✅ SpikeWeightedMSELoss — baseline=1.0x, spike=5.0x penalty
✅ Training engine ready


In [ ]:
# CELL 7 — TRAIN
history = train_tcn(model, tcn_tr, tcn_vl, n_epochs=120, lr=3e-4, patience=20)


──────────────────────────────────────────────────────────────────────
  TCN v4  |  4.9s → 4.9s  |  491,136 params
  MultiScale encoder | U-Net decoder | SpikeWeightedMSE | batch=64
──────────────────────────────────────────────────────────────────────


Epochs:   0%|                                                     | 0/120 [00:00<?, ?ep/s]

Train:   0%|                                                      | 0/490 [00:00<?, ?it/s]

    1 | train=4.86490  val=4.79446  lr=3.00e-04  ★


Train:   0%|                                                      | 0/490 [00:00<?, ?it/s]

...

    3 | train=4.75036  val=4.78082  lr=3.00e-04  ★


Train:   0%|                                                      | 0/490 [00:00<?, ?it/s]

Train:   0%|                                                      | 0/490 [00:00<?, ?it/s]

    5 | train=4.70317  val=4.77742  lr=2.99e-04  ★


Train:   0%|                                                      | 0/490 [00:00<?, ?it/s]

In [ ]:
# CELL 8 — TRAINING HISTORY
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ep = range(1, len(history['train_loss'])+1)

axes[0].plot(ep, history['train_loss'], color='#0ea5e9', lw=2, label='Train', marker='o', ms=3)
axes[0].plot(ep, history['val_loss'],   color='#ef4444', lw=2, label='Val',   marker='s', ms=3, ls='--')
best_ep = int(np.argmin(history['val_loss']))+1
axes[0].axvline(best_ep, color='gold', ls=':', lw=2, label=f'Best ep {best_ep}')
axes[0].set_title(f'TCN v4 Loss ({INPUT_LEN/FS:.1f}s→{HORIZON/FS:.1f}s)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('SpikeWeighted MSE')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(ep, history['lr'], color='#10b981', lw=2, marker='o', ms=3)
axes[1].set_title('Learning Rate (CosineAnnealing)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('LR')
axes[1].set_yscale('log'); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '01_training_history.png'), dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved 01_training_history.png")

In [ ]:
# CELL 9 — TEST EVALUATION
test_loss, test_preds, test_targets = eval_epoch(model, tcn_te, DEVICE)

mae_per_lead, rmse_per_lead = [], []
for i in range(N_LEADS):
    p = test_preds[:, :, i].flatten()
    t = test_targets[:, :, i].flatten()
    mae_per_lead.append(mean_absolute_error(t, p))
    rmse_per_lead.append(np.sqrt(mean_squared_error(t, p)))

print(f"Test SpikeWeightedMSE : {test_loss:.6f}")
print(f"\nPer-Lead MAE:")
for i, name in enumerate(LEAD_NAMES):
    print(f"  {name:>4s}: MAE={mae_per_lead[i]:.4f}  RMSE={rmse_per_lead[i]:.4f}")
print(f"\nMacro MAE  : {np.mean(mae_per_lead):.6f}")
print(f"Macro RMSE : {np.mean(rmse_per_lead):.6f}")
print("✅ Evaluation complete")

In [ ]:
# CELL 10 — PER-LEAD RMSE
fig, ax = plt.subplots(figsize=(13, 6))
colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, N_LEADS))
bars   = ax.bar(np.arange(N_LEADS), rmse_per_lead, width=0.65,
                color=colors, alpha=0.85, edgecolor='black')
ax.axhline(np.mean(rmse_per_lead), color='#f59e0b', ls='--', lw=2.5,
           label=f'Macro RMSE: {np.mean(rmse_per_lead):.4f}')
for bar, val in zip(bars, rmse_per_lead):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height(),
            f'{val:.4f}', ha='center', va='bottom', fontsize=9)
ax.set_xticks(np.arange(N_LEADS)); ax.set_xticklabels(LEAD_NAMES, fontsize=12, fontweight='bold')
ax.set_ylabel('RMSE (mV)', fontsize=12)
ax.set_title(f'TCN v4 Per-Lead RMSE ({INPUT_LEN/FS:.1f}s→{HORIZON/FS:.1f}s)',
             fontsize=14, fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '02_per_lead_rmse.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# CELL 11 — PREDICTED vs ACTUAL
n_samples    = 4
lead_indices = [0, 1, 2, 6]
t = np.arange(HORIZON) / FS

fig, axes = plt.subplots(n_samples, 4, figsize=(20, 14))
for row in range(n_samples):
    for col, lead_idx in enumerate(lead_indices):
        ax = axes[row, col]
        ax.plot(t, test_targets[row, :, lead_idx],
                color='#0ea5e9', lw=2, label='Actual', alpha=0.85)
        ax.plot(t, test_preds[row, :, lead_idx],
                color='#ef4444', lw=1.5, label='Predicted', ls='--', alpha=0.85)
        rmse_i = np.sqrt(mean_squared_error(
            test_targets[row,:,lead_idx], test_preds[row,:,lead_idx]))
        ax.set_title(f'{LEAD_NAMES[lead_idx]}  RMSE={rmse_i:.4f}',
                     fontsize=10, fontweight='bold')
        ax.set_xlabel('Time (s)'); ax.set_ylabel('mV'); ax.grid(True, alpha=0.3)
        if col == 0: ax.legend(fontsize=8)

fig.suptitle(f'TCN v4: Predicted vs Actual — {HORIZON/FS:.1f}s horizon',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '03_predictions_overlay.png'), dpi=150, bbox_inches='tight')
plt.show()
print("✅ Overlay saved")

In [ ]:
# CELL 12 — SPIKE TRACKING ANALYSIS
spike_mae, base_mae = [], []
for i in range(N_LEADS):
    actual = test_targets[:, :, i]
    pred   = test_preds[:,   :, i]
    thresh = actual.std() * 1.5
    sm = np.abs(actual) > thresh
    bm = ~sm
    if sm.sum() > 0: spike_mae.append(np.abs(actual[sm] - pred[sm]).mean())
    if bm.sum() > 0: base_mae.append( np.abs(actual[bm] - pred[bm]).mean())

spike_mean = np.mean(spike_mae)
base_mean  = np.mean(base_mae)
ratio      = spike_mean / base_mean

print(f"Baseline MAE (flat regions) : {base_mean:.4f} mV")
print(f"Spike MAE   (|sig| > 1.5σ)  : {spike_mean:.4f} mV")
print(f"Spike/base ratio            : {ratio:.2f}x")
if   ratio < 2.5: print("✅ Excellent spike tracking")
elif ratio < 4.0: print("✅ Good spike tracking")
elif ratio < 6.0: print("⚠  Partial spike tracking")
else:             print("❌ Spikes still missed")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

x = np.arange(N_LEADS)
bm_l, sm_l = [], []
for i in range(N_LEADS):
    a = test_targets[:,:,i]; p = test_preds[:,:,i]
    thr = a.std()*1.5
    sm  = np.abs(a) > thr; bm = ~sm
    sm_l.append(np.abs(a[sm]-p[sm]).mean() if sm.sum()>0 else 0)
    bm_l.append(np.abs(a[bm]-p[bm]).mean() if bm.sum()>0 else 0)

axes[0].bar(x-0.2, bm_l, 0.4, label='Baseline MAE',
            color='#0ea5e9', alpha=0.85, edgecolor='black')
axes[0].bar(x+0.2, sm_l, 0.4, label='Spike MAE',
            color='#ef4444', alpha=0.85, edgecolor='black')
axes[0].set_xticks(x); axes[0].set_xticklabels(LEAD_NAMES, fontsize=10, fontweight='bold')
axes[0].set_ylabel('MAE (mV)'); axes[0].legend()
axes[0].set_title('Baseline vs Spike MAE', fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='y')

af = test_targets[:,:,1].flatten(); pf = test_preds[:,:,1].flatten()
idx = np.random.choice(len(af), 5000, replace=False)
axes[1].scatter(af[idx], pf[idx], alpha=0.15, s=4, color='#10b981')
mn = min(af.min(), pf.min()); mx = max(af.max(), pf.max())
axes[1].plot([mn,mx],[mn,mx],'r--',lw=2,label='Perfect')
axes[1].set_xlabel('Actual (mV)'); axes[1].set_ylabel('Predicted (mV)')
axes[1].set_title('Actual vs Predicted — Lead II', fontweight='bold')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '05_spike_tracking.png'), dpi=150, bbox_inches='tight')
plt.show()
print("✅ Spike analysis saved")

In [ ]:
# CELL 13 — SAVE RESULTS
results = {
    'model': 'TCN_v4_UNet', 'horizon_s': HORIZON/FS, 'input_s': INPUT_LEN/FS,
    'n_parameters': n_params, 'test_loss': test_loss,
    'mae_per_lead': mae_per_lead, 'mae_macro': np.mean(mae_per_lead),
    'rmse_per_lead': rmse_per_lead, 'rmse_macro': np.mean(rmse_per_lead),
    'history': history, 'lead_names': LEAD_NAMES,
    'test_preds': test_preds, 'test_targets': test_targets,
}
path = os.path.join(CKPT_DIR, 'TCN_results_summary.pkl')
with open(path, 'wb') as f: pickle.dump(results, f)

print(f"{'='*60}")
print(f"  TCN v4 FINAL RESULTS  ({INPUT_LEN/FS:.1f}s → {HORIZON/FS:.1f}s)")
print(f"{'='*60}")
print(f"  Parameters : {n_params:,}")
print(f"  Test Loss  : {test_loss:.6f}")
print(f"  Macro MAE  : {np.mean(mae_per_lead):.6f} mV")
print(f"  Macro RMSE : {np.mean(rmse_per_lead):.6f} mV")
print(f"  Saved      → {path}")
print("✅ TCN v4 Complete!")